[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exploring-yash/seamless_interaction_dynamics/blob/main/Notebooks/seamless_data_pipeline.ipynb)

# Seamless Interaction Data Pipeline

**A balanced, reproducible pipeline for downloading behavioral signals from Meta's Seamless Interaction dataset.**

This notebook downloads, validates, and organizes NPZ feature files from [Meta's Seamless Interaction dataset](https://huggingface.co/datasets/facebook/seamless-interaction) — the largest publicly available record of human dyadic interaction (16,267 conversations, 4,285 participants, 30 Hz).

**What it does:**
1. Downloads movement features (emotion, FAU, gaze, head, body) from [Hugging Face](https://huggingface.co/datasets/facebook/seamless-interaction) archives
2. Samples **balanced stranger/familiar dyads** using [relationships.csv](https://github.com/facebookresearch/seamless_interaction/blob/main/assets/relationships.csv)
3. Optionally downloads audio (.wav) and video (.mp4)
4. Validates data quality and saves a reproducibility manifest

**Dataset links:**
- Hugging Face: [facebook/seamless-interaction](https://huggingface.co/datasets/facebook/seamless-interaction)
- GitHub (Meta): [facebookresearch/seamless_interaction](https://github.com/facebookresearch/seamless_interaction)
- License: [BSD-style](https://huggingface.co/datasets/facebook/seamless-interaction/blob/main/LICENSE)

**Requirements:** Google Colab (recommended), ~12 GB Drive space, [Hugging Face account](https://huggingface.co/join) with [dataset access](https://huggingface.co/datasets/facebook/seamless-interaction) approved.

In [1]:
# ============================================================
# 1. Environment Setup
# ============================================================
import os
import sys
import json
import time
import warnings
import logging
import re
import hashlib
import tempfile
import urllib.request
import urllib.error
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('pipeline')

# Core Dataset Constants
SAMPLE_RATE_HZ = 30  # Movement features: 30 Hz (video frame rate)

# Detect environment and mount Google Drive
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        DRIVE_ROOT = Path('/content/drive/MyDrive/seamless_interaction')
        IN_COLAB = True
        print('✓ Google Drive mounted')
    except Exception as mount_err:
        print(f'⚠ Google Drive mount failed: {mount_err}')
        print('  Falling back to local /tmp directory')
        DRIVE_ROOT = Path('/tmp/seamless_interaction')
        IN_COLAB = False
except ImportError:
    print('⚠ Not in Colab. Using local directory.')
    DRIVE_ROOT = Path('/tmp/seamless_interaction')
    IN_COLAB = False

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'✓ Data root: {DRIVE_ROOT}')

# Clone and install Seamless Interaction repository
REPO_DIR = DRIVE_ROOT / 'seamless_interaction_repo'
if not (REPO_DIR / '.git').exists():
    print('Cloning seamless_interaction repository...')
    os.system(f'git clone https://github.com/facebookresearch/seamless_interaction.git {REPO_DIR} > /dev/null 2>&1')
    os.system(f'pip install -e {REPO_DIR} -q > /dev/null 2>&1')
    os.system('pip install wget -q > /dev/null 2>&1')
    print(f'✓ Seamless Interaction installed')
else:
    print(f'✓ Seamless Interaction repo already present')
    # Ensure wget is installed (needed by SeamlessInteractionFS)
    os.system('pip install wget -q > /dev/null 2>&1')

# Add to path and import
sys.path.insert(0, str(REPO_DIR / 'src'))
from seamless_interaction.fs import SeamlessInteractionFS, DatasetConfig
print('✓ SeamlessInteractionFS ready')

print('\n✓ Environment setup complete')



Mounted at /content/drive
✓ Google Drive mounted
✓ Data root: /content/drive/MyDrive/seamless_interaction
✓ Seamless Interaction repo already present
✓ SeamlessInteractionFS ready

✓ Environment setup complete


## The Cast List: Configuration & Metadata

We're working with 4 CSV files that form the complete index (sourced from [facebookresearch/seamless_interaction/assets/](https://github.com/facebookresearch/seamless_interaction/tree/main/assets)):
- `interactions.csv`: 16,267 conversations with prompts and context
- `participants.csv`: Demographic information for 4,285 individuals  
- `relationships.csv`: The relationship between each pair (stranger, friends, coworkers, etc.)
- `filelist.csv`: The map to behavioral data in Meta's S3 storage

These CSVs act as the "cast list"—they provide details on the speakers, their assigned discussion topics, and the storage locations of their conversations.

In [2]:
# ============================================================
# 2. Pipeline Configuration
# ============================================================

# --- Data Selection ---
TARGET_DYADS = 500          # Headroom for 436 actual dyads (311 naturalistic + 125 improvised)
TARGET_GB = 25.0            # NPZ features budget (~8.7 GB actual for 436 dyads)
SEED = 42                   # Reproducibility

# --- Target Data Slice (Multi-Condition) ---
# V00 is the ONLY vendor with rich movement features (emotion, FAU, gaze, head).
# Improvised sessions use professional actors = always strangers (0 familiar).
# Naturalistic sessions have real-world pairs = both familiar AND stranger.
#
# Strategy: Download BOTH conditions for a hybrid analysis:
#   - naturalistic/dev: familiar + stranger (clean H1 comparison, same prompt type)
#   - improvised/dev: stranger only (actor baseline, separate condition)
# Class imbalance within naturalistic (~5:1 familiar:stranger) is handled
# at the classifier level via class_weight='balanced' + stratified CV.

TARGET_LABELS = ['naturalistic', 'improvised']   # Conditions to download
TARGET_SPLIT = 'dev'                              # 'train', 'dev', or 'test'

# Per-condition configuration
CONDITION_CONFIG = {
    'naturalistic': {
        'max_dyads': None,        # Download ALL available
        'role': 'primary',        # Used for H1 classification
    },
    'improvised': {
        'max_dyads': 125,         # Actor baseline sample
        'role': 'baseline',       # Separate condition, NOT mixed into H1
    },
}

# --- Relationship Validation ---
# Ensures the primary condition has both stranger AND familiar dyads.
# Raises ValueError if either class is empty (prevents useless downloads).
BALANCE_RELATIONSHIPS = True
EXCLUDE_UNKNOWN = True          # Hard-exclude dyads with no relationship label

# Backwards compat: accept a single string
if isinstance(TARGET_LABELS, str):
    TARGET_LABELS = [TARGET_LABELS]

# --- Feature Tiers (must match S3 path names exactly) ---
# S3 URL: {base}/{label}/{split}/{feature_group}/{feature}/{file_id}.npy
# Each feature is a separate .npy file, NOT bundled .npz

CORE_FEATURES = {
    "movement": ["emotion_valence", "emotion_arousal", "is_valid"],
}

RICH_FEATURES = {
    "movement": [
        "emotion_valence",
        "emotion_arousal",
        "is_valid",
        "expression",
        "FAUValue",
        "gaze_encodings",
        "emotion_scores",
    ],
}

FULL_FEATURES = {
    "movement": [
        "emotion_valence",
        "emotion_arousal",
        "is_valid",
        "expression",
        "FAUValue",
        "gaze_encodings",
        "emotion_scores",
        "head_encodings",
        "alignment_head_rotation",
        "frame_latent",
        "hypernet_features",
        "EmotionArousalToken",
        "EmotionValenceToken",
        "FAUToken",
        "alignment_translation",
    ],
    "smplh": [
        "body_pose",
        "global_orient",
        "is_valid",
        "left_hand_pose",
        "right_hand_pose",
        "translation",
    ],
}

# Select tier — FULL recommended for dyadic coupling analysis
# (needs head_encodings for WCC and smplh:translation for motion energy)
SELECTED_FEATURES = FULL_FEATURES
FEATURE_TIER_NAME = "Full"

# --- S3 Configuration ---
S3_BASE = "https://dl.fbaipublicfiles.com/seamless_interaction"
HF_REPO = "facebook/seamless-interaction"
HF_RAW_BASE = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main"
MAX_WORKERS = 4             # Parallel download threads (modest for Colab stability)
RETRY_COUNT = 3             # Retries per failed download
RETRY_DELAY = 2             # Seconds between retries

# --- Audio Download ---
DOWNLOAD_AUDIO = True               # Toggle audio download on/off
AUDIO_BUDGET_GB = 50.0              # Budget for audio (3.84 TB free — no need to be tight)
AUDIO_SAMPLE_RATE = 48000           # 48 kHz
AUDIO_BITS = 16                     # 16-bit PCM

# --- Video Download ---
DOWNLOAD_VIDEO = True               # Toggle video download on/off
VIDEO_BUDGET_GB = 100.0             # Budget for video files (3.84 TB free)
# NOTE: Videos are inside tar archives on S3, not individual files.
# The pipeline downloads tars, extracts .mp4s for target file_ids, then deletes the tar.

# --- Derived ---
np.random.seed(SEED)

print('Configuration:')
print(f'  Target: {TARGET_DYADS} dyads | NPZ budget: {TARGET_GB} GB')
print(f'  Conditions: {TARGET_LABELS} / {TARGET_SPLIT}')
for lbl in TARGET_LABELS:
    cfg = CONDITION_CONFIG.get(lbl, {})
    print(f'    {lbl}: role={cfg.get("role", "?")} max_dyads={cfg.get("max_dyads", "all")}')
print(f'  Feature tier: {FEATURE_TIER_NAME}')
feature_count = sum(len(v) for v in SELECTED_FEATURES.values())
print(f'  Features per file: {feature_count} across {len(SELECTED_FEATURES)} groups')
print(f'  Relationship validation: {BALANCE_RELATIONSHIPS} (requires both classes in primary)')
print(f'  Audio download: {"ON" if DOWNLOAD_AUDIO else "OFF"} (budget: {AUDIO_BUDGET_GB} GB)')
print(f'  Video download: {"ON" if DOWNLOAD_VIDEO else "OFF"} (budget: {VIDEO_BUDGET_GB} GB)')


Configuration:
  Target: 500 dyads | NPZ budget: 25.0 GB
  Conditions: ['naturalistic', 'improvised'] / dev
    naturalistic: role=primary max_dyads=None
    improvised: role=baseline max_dyads=125
  Feature tier: Full
  Features per file: 21 across 2 groups
  Relationship validation: True (requires both classes in primary)
  Audio download: ON (budget: 50.0 GB)
  Video download: ON (budget: 100.0 GB)


## Step 1: Download the Index

The metadata CSVs are small (~1 MB total), but they unlock the entire dataset. We'll load them from Hugging Face and verify that every interaction has a corresponding video in S3.

In [4]:
# ============================================================
# 3. Download Metadata CSVs from Hugging Face
# ============================================================

filelist_df = None  # Initialized here; assigned below if filelist is found

CSV_FILES = ['interactions.csv', 'participants.csv', 'relationships.csv']
FILELIST_URLS = [
    f"{HF_RAW_BASE}/filelist.csv",
    f"{HF_RAW_BASE}/assets/filelist.csv",
]

print('⬇ Fetching metadata CSVs...')

# Download main CSVs
for csv_name in CSV_FILES:
    target = DRIVE_ROOT / csv_name
    if target.exists():
        size_kb = target.stat().st_size / 1024
        print(f'  ✓ {csv_name} exists ({size_kb:.0f} KB)')
        continue

    url = f"{HF_RAW_BASE}/{csv_name}"
    print(f'  ⬇ {csv_name}...', end=' ', flush=True)
    try:
        urllib.request.urlretrieve(url, str(target))
        size_kb = target.stat().st_size / 1024
        print(f'✓ ({size_kb:.0f} KB)')
    except Exception as e:
        print(f'✗ {e}')

# Try to download filelist.csv from multiple locations
filelist_path = DRIVE_ROOT / 'filelist.csv'
filelist_found = False

if not filelist_path.exists():
    print(f'  ⬇ filelist.csv...', end=' ', flush=True)
    for url in FILELIST_URLS:
        try:
            urllib.request.urlretrieve(url, str(filelist_path))
            size_kb = filelist_path.stat().st_size / 1024
            print(f'✓ ({size_kb:.0f} KB)')
            filelist_found = True
            break
        except urllib.error.HTTPError as e:
            if e.code != 404:
                print(f'⚠ {url}: {e}')
        except Exception:
            pass

    if not filelist_found:
        print('⚠ filelist.csv not found on Hugging Face')
        print('  Will use fallback strategy: probe S3 systematically')
else:
    filelist_found = True
    size_kb = filelist_path.stat().st_size / 1024
    print(f'  ✓ filelist.csv exists ({size_kb:.0f} KB)')

# Load metadata
print('\n📖 Loading metadata...')
interactions_df = pd.read_csv(DRIVE_ROOT / 'interactions.csv')
participants_df = pd.read_csv(DRIVE_ROOT / 'participants.csv')
relationships_df = pd.read_csv(DRIVE_ROOT / 'relationships.csv')

filelist_df = None
if filelist_found and filelist_path.exists():
    try:
        filelist_df = pd.read_csv(filelist_path)
        print(f'✓ Metadata loaded:')
        print(f'  Interactions: {len(interactions_df):,} rows')
        print(f'  Participants: {len(participants_df):,} rows')
        print(f'  Relationships: {len(relationships_df):,} rows')
        print(f'  File list: {len(filelist_df):,} rows')
    except Exception as e:
        print(f'✗ Failed to load filelist.csv: {e}')
        filelist_df = None
else:
    print(f'✓ Metadata loaded:')
    print(f'  Interactions: {len(interactions_df):,} rows')
    print(f'  Participants: {len(participants_df):,} rows')
    print(f'  Relationships: {len(relationships_df):,} rows')
    print(f'  File list: NOT AVAILABLE (will use fallback)')

⬇ Fetching metadata CSVs...
  ✓ interactions.csv exists (1437 KB)
  ✓ participants.csv exists (304 KB)
  ✓ relationships.csv exists (138 KB)
  ⬇ filelist.csv... ⚠ filelist.csv not found on Hugging Face
  Will use fallback strategy: probe S3 systematically

📖 Loading metadata...
✓ Metadata loaded:
  Interactions: 1,312 rows
  Participants: 4,284 rows
  Relationships: 5,098 rows
  File list: NOT AVAILABLE (will use fallback)


In [5]:
# ============================================================
# 4. Explore Dataset & Select Target Interactions
# ============================================================

target_interactions = interactions_df.copy()

print(f'📊 Dataset Overview:')
print(f'  Total interactions: {len(target_interactions):,}')

# Inspect structure
print(f'\n  CSV Columns: {list(target_interactions.columns)}')
print(f'\n  Sample row (first 5 columns):')
for col in target_interactions.columns[:5]:
    val = target_interactions.iloc[0][col]
    # Truncate long text values for readability
    if isinstance(val, str) and len(str(val)) > 60:
        val = str(val)[:57] + '...'
    print(f'    {col}: {val}')

# Show distribution of interaction types if available
if 'interaction_type' in target_interactions.columns:
    types = target_interactions['interaction_type'].value_counts()
    print(f'\n  Interaction Types:')
    for itype, count in types.items():
        print(f'    {itype}: {count}')

📊 Dataset Overview:
  Total interactions: 1,312

  CSV Columns: ['Unnamed: 0', 'prompt_hash', 'prompt_id_unique', 'participant_a_prompt_text', 'participant_b_prompt_text', 'ipc_a', 'ipc_b', 'interaction_type']

  Sample row (first 5 columns):
    Unnamed: 0: 0
    prompt_hash: 0
    prompt_id_unique: P0070_v3.4.fam_ANCP_XXXX
    participant_a_prompt_text: Tell your partner about a time when you went out of your ...
    participant_b_prompt_text: Your partner will tell you about a time they made an effo...

  Interaction Types:
    ipc_conversation: 946
    grounded_gesture: 322
    collaborative_storytelling: 40
    charades: 4


In [6]:
# ============================================================
# 5. Build File ID Registry & Select Dyads (Multi-Condition, Relationship-Balanced)
# ============================================================

# Relationship categories from relationships.csv 'relationship' column
FAMILIAR_LABELS = {'familiar', 'friends', 'coworkers', 'family-generic',
                   'familiar-generic', 'siblings', 'parent_child',
                   'dating', 'spouse', 'romantic_partner', 'neighbors'}

def get_relationship_for_dyad(file_ids):
    """Extract (vendor_id, session_id) from file_id and look up relationship."""
    fid = file_ids[0]
    parts = fid.split('_')
    if len(parts) < 2:
        return 'unknown'
    vid = parts[0]                    # 'V00'
    try:
        sid = int(parts[1][1:])       # 'S0062' -> 62
    except (ValueError, IndexError):
        return 'unknown'
    match = relationships_df[
        (relationships_df['vendor_id'] == vid) &
        (relationships_df['session_id'] == sid)
    ]
    if len(match) > 0:
        return match.iloc[0]['relationship'].strip().lower()
    return 'unknown'

# ---- Step 1: Query each condition separately ----
all_condition_dyads = {}     # {condition: {interaction_key: [file_ids]}}
all_dyad_metadata = {}       # {file_id: {'label': ..., 'split': ..., 'condition': ...}}

# V00 is the ONLY vendor with rich movement features (emotion, FAU, gaze, head).
# get_available_keys() ignores preferred_vendors_only and returns ALL vendors,
# so we must filter to V00 explicitly.
V00_PREFIX = 'V00'

for condition_label in TARGET_LABELS:
    print(f'\n--- Querying {condition_label}/{TARGET_SPLIT} ---')

    cfg = DatasetConfig(
        label=condition_label,
        split=TARGET_SPLIT,
        preferred_vendors_only=True,
        local_dir=str(DRIVE_ROOT),
    )
    fs = SeamlessInteractionFS(config=cfg)

    file_ids = fs.get_available_keys(
        key_type='file',
        label=condition_label,
        split=TARGET_SPLIT,
    )

    # Filter to V00 only — the only vendor with rich movement features
    file_ids = [fid for fid in file_ids if fid.startswith(V00_PREFIX)]
    print(f'  Found {len(file_ids)} V00 file_ids (filtered from all vendors)')

    # Group into dyads
    dyad_groups = {}
    for fid in file_ids:
        parts = fid.rsplit('_', 1)
        if len(parts) == 2:
            ikey = parts[0]
            dyad_groups.setdefault(ikey, []).append(fid)
            all_dyad_metadata[fid] = {
                'label': condition_label,
                'split': TARGET_SPLIT,
                'condition': condition_label,
            }

    complete = {k: v for k, v in dyad_groups.items() if len(v) == 2}
    print(f'  Complete dyads (2 participants): {len(complete)}')
    all_condition_dyads[condition_label] = complete

# ---- Step 2: Classify relationships per condition ----
# Unknowns are excluded implicitly: only 'stranger' and FAMILIAR_LABELS
# entries enter the pools. Dyads with 'unknown' relationship are never sampled.
condition_classified = {}

for condition_label, complete in all_condition_dyads.items():
    strangers, familiars, unknowns = [], [], []

    for ikey, fids in complete.items():
        rel = get_relationship_for_dyad(fids)
        if rel == 'stranger':
            strangers.append(ikey)
        elif rel in FAMILIAR_LABELS:
            familiars.append(ikey)
        else:
            unknowns.append(ikey)

    condition_classified[condition_label] = {
        'stranger': strangers,
        'familiar': familiars,
        'unknown': unknowns,
    }

    print(f'  {condition_label}: {len(strangers)} stranger, {len(familiars)} familiar, {len(unknowns)} excluded (unknown)')

# Vendor coverage check
for condition_label, complete in all_condition_dyads.items():
    if len(complete) > 0:
        sample_vid = list(complete.values())[0][0].split('_')[0]
        vendor_in_rel = relationships_df[relationships_df['vendor_id'] == sample_vid]
        if len(vendor_in_rel) == 0:
            print(f'\n  WARNING: relationships.csv has NO {sample_vid} entries for {condition_label}!')

# ---- Step 3: Per-condition sampling ----
sampled_keys = []              # Final list of interaction keys to download
sampled_condition_map = {}     # {interaction_key: condition_label}

for condition_label in TARGET_LABELS:
    cond_cfg = CONDITION_CONFIG.get(condition_label, {})
    classified = condition_classified[condition_label]
    pool_strangers = classified['stranger']
    pool_familiars = classified['familiar']

    if cond_cfg.get('role') == 'primary':
        # --- PRIMARY CONDITION: take ALL labeled dyads ---
        # Class imbalance is handled at classifier level (class_weight='balanced',
        # stratified k-fold CV) — not by discarding data.
        if BALANCE_RELATIONSHIPS and len(pool_familiars) == 0:
            raise ValueError(
                f"FATAL: {condition_label}/{TARGET_SPLIT} has 0 familiar dyads. "
                f"Cannot run H1 classification. If targeting 'improvised', actors are "
                f"always strangers. Add 'naturalistic' to TARGET_LABELS for familiar dyads."
            )
        if BALANCE_RELATIONSHIPS and len(pool_strangers) == 0:
            raise ValueError(
                f"FATAL: {condition_label}/{TARGET_SPLIT} has 0 stranger dyads. "
                f"Cannot run H1 classification."
            )

        condition_sampled = pool_strangers + pool_familiars
        print(f'\n  {condition_label} (primary): {len(pool_strangers)} stranger + {len(pool_familiars)} familiar = {len(condition_sampled)} dyads')
        print(f'    Imbalance handled via class_weight="balanced" + stratified CV downstream')

    elif cond_cfg.get('role') == 'baseline':
        # --- BASELINE CONDITION: sample up to max_dyads, no balance ---
        max_d = cond_cfg.get('max_dyads') or (len(pool_strangers) + len(pool_familiars))
        pool = pool_strangers + pool_familiars
        if len(pool) > max_d:
            condition_sampled = list(np.random.choice(pool, max_d, replace=False))
        else:
            condition_sampled = pool
        print(f'\n  {condition_label} (baseline): {len(condition_sampled)} dyads (max {max_d})')

    else:
        condition_sampled = pool_strangers + pool_familiars
        print(f'\n  {condition_label}: {len(condition_sampled)} dyads')

    for k in condition_sampled:
        sampled_keys.append(k)
        sampled_condition_map[k] = condition_label

np.random.shuffle(sampled_keys)

# ---- Step 4: Build final sampled_dyads dict ----
sampled_dyads = {}
for k in sampled_keys:
    for condition_label, dyads_dict in all_condition_dyads.items():
        if k in dyads_dict:
            sampled_dyads[k] = dyads_dict[k]
            break

total_files = sum(len(v) for v in sampled_dyads.values())

print(f'\n{"="*60}')
print(f'Final Dyad Selection:')
print(f'  Total: {len(sampled_dyads)} dyads ({total_files} files)')

# Per-condition summary
for condition_label in TARGET_LABELS:
    cond_keys = [k for k in sampled_keys if sampled_condition_map.get(k) == condition_label]
    classified = condition_classified.get(condition_label, {})
    s_count = sum(1 for k in cond_keys if k in classified.get('stranger', []))
    f_count = sum(1 for k in cond_keys if k in classified.get('familiar', []))
    print(f'  {condition_label}: {len(cond_keys)} dyads (stranger={s_count}, familiar={f_count})')

if sampled_dyads:
    sample_key = list(sampled_dyads.keys())[0]
    cond = sampled_condition_map.get(sample_key, '?')
    print(f'\n  Sample dyad ({cond}): {sample_key}')
    for fid in sampled_dyads[sample_key]:
        print(f'    -> {fid}')



--- Querying naturalistic/dev ---
  Found 622 V00 file_ids (filtered from all vendors)
  Complete dyads (2 participants): 311

--- Querying improvised/dev ---
  Found 340 V00 file_ids (filtered from all vendors)
  Complete dyads (2 participants): 170
  naturalistic: 52 stranger, 259 familiar, 0 excluded (unknown)
  improvised: 170 stranger, 0 familiar, 0 excluded (unknown)

  naturalistic (primary): 52 stranger + 259 familiar = 311 dyads
    Imbalance handled via class_weight="balanced" + stratified CV downstream

  improvised (baseline): 125 dyads (max 125)

Final Dyad Selection:
  Total: 436 dyads (872 files)
  naturalistic: 311 dyads (stranger=52, familiar=259)
  improvised: 125 dyads (stranger=125, familiar=0)

  Sample dyad (naturalistic): V00_S0553_I00000307
    -> V00_S0553_I00000307_P0466
    -> V00_S0553_I00000307_P0696


In [7]:
# ============================================================
# 6. Estimate Download Size & Adjust Budget
# ============================================================

# Size estimates based on feature tier
est_mb_per_file = {
    "Core": 0.12,     # 120 KB per file (3 features)
    "Rich": 8.5,      # 8.5 MB per file (7 features, some high-dim)
    "Full": 22.0,     # 22 MB per file (15 movement + 6 SMPL-H features)
}.get(FEATURE_TIER_NAME, 8.5)

total_files = sum(len(v) for v in sampled_dyads.values())
est_total_gb = (total_files * est_mb_per_file) / 1024

print(f'Size Estimate:')
print(f'  ~{est_mb_per_file:.1f} MB per file x {total_files} files')
print(f'  Estimated total: {est_total_gb:.1f} GB')

# Adjust sample if needed to fit budget
if est_total_gb > TARGET_GB * 1.2 and total_files > 0:
    print(f'\n  Exceeds budget ({est_total_gb:.1f} > {TARGET_GB*1.2:.1f} GB). Adjusting...')
    max_files = int((TARGET_GB * 1024) / est_mb_per_file)
    max_dyads = max(1, max_files // 2)

    if len(sampled_keys) > max_dyads:
        # Proportional reduction per condition
        from collections import Counter
        cond_counts = Counter(sampled_condition_map[k] for k in sampled_keys)
        ratio = max_dyads / len(sampled_keys)

        new_sampled = []
        for cond, count in cond_counts.items():
            cond_keys = [k for k in sampled_keys if sampled_condition_map.get(k) == cond]
            n_keep = max(1, int(count * ratio))
            new_sampled.extend(list(np.random.choice(cond_keys, n_keep, replace=False)))

        sampled_keys = new_sampled
        sampled_dyads = {k: sampled_dyads[k] for k in sampled_keys if k in sampled_dyads}
        total_files = sum(len(v) for v in sampled_dyads.values())
        est_total_gb = (total_files * est_mb_per_file) / 1024
        print(f'  Adjusted to {len(sampled_dyads)} dyads ({est_total_gb:.1f} GB)')
else:
    print(f'  Within budget')

if total_files > 0:
    print(f'\n  Time Estimate:')
    min_time = (total_files * 8) / 60
    max_time = (total_files * 15) / 60
    print(f'  {min_time:.0f}-{max_time:.0f} minutes ({min_time/60:.1f}-{max_time/60:.1f} hours)')


Size Estimate:
  ~22.0 MB per file x 872 files
  Estimated total: 18.7 GB
  Within budget

  Time Estimate:
  116-218 minutes (1.9-3.6 hours)


## Step 2: The Feature Extraction Strategy

The [Seamless Interaction dataset](https://huggingface.co/datasets/facebook/seamless-interaction) stores features as `.npy` files on S3, bundled into `.tar` archives on Hugging Face. Each participant file contains up to 21 behavioral channels at 30 Hz:

| Group | Features | Dimensions |
|-------|----------|-----------|
| `movement` | emotion_valence, emotion_arousal, FAUValue (24), gaze_encodings (2), head_encodings (3), expression (128), emotion_scores (8) | ~166 dims |
| `smplh` | body_pose (63), translation (3), global_orient (3), hand poses | ~75 dims |

Together: a highly multi-dimensional behavioral state vector (over 240 dimensions) sampled 30 times per second — the frequency needed to detect emotional synchrony at human scale (~500 ms to 2 second windows).

See [Meta's dataset documentation](https://github.com/facebookresearch/seamless_interaction#data-format) for the full feature specification.

In [8]:
# ============================================================
# 7. Feature Download Functions (Hugging Face-first strategy)
# ============================================================
#
# Meta's S3 (dl.fbaipublicfiles.com) returns 403 for individual .npy files.
# The working path is Hugging Face tar archives → extract → load .npy files.
#
# Strategy:
#   1. Group file_ids by (batch_idx, archive_idx)
#   2. Download each tar archive ONCE from Hugging Face
#   3. Extract and load .npy files for all file_ids in that archive
#   4. S3 as fallback only (gather_file_id_data_from_s3)

def download_hf_archive_if_needed(fs, file_id):
    """
    Download the Hugging Face tar archive containing this file_id (if not already extracted).
    Returns the extraction directory path, or None on failure.
    """
    try:
        entry = fs._cached_filelist[fs._cached_filelist['file_id'] == file_id].iloc[0]
        batch_idx = int(entry['batch_idx'])
        archive_idx = int(entry['archive_idx'])
        label = entry['label']
        split = entry['split']

        # Check if already extracted
        extract_dir = Path(fs.config.local_dir) / label / split / f'{batch_idx:04d}' / f'{archive_idx:04d}'
        if extract_dir.exists() and any(extract_dir.rglob('*.npy')):
            return str(extract_dir)

        # Download and extract the tar archive
        print(f'    ⬇ Downloading HF archive: {label}/{split}/{batch_idx:04d}/{archive_idx:04d}.tar ...')
        success, result_path = fs.download_archive_from_hf(
            archive=archive_idx,
            label=label,
            split=split,
            batch=batch_idx,
            extract=True,
        )

        if success:
            return result_path
        else:
            print(f'    ✗ HF archive download failed. Path: {result_path}')
            print(f'    → You may need a Hugging Face token: huggingface-cli login')
            print(f'    → Or accept dataset terms at: https://huggingface.co/datasets/facebook/seamless-interaction')
            return None
    except Exception as e:
        print(f'    ✗ HF archive error: {type(e).__name__}: {e}')
        return None


def load_features_from_extracted_npy(file_id, extract_dir, features_dict=SELECTED_FEATURES):
    """
    Load selected features from an extracted Hugging Face archive.

    HF tars use WebDataset format — each tar contains FLAT files per file_id:
      {extract_dir}/{file_id}.npz   ← consolidated features (movement:emotion_valence, etc.)
      {extract_dir}/{file_id}.json  ← metadata
      {extract_dir}/{file_id}.wav   ← audio
      {extract_dir}/{file_id}.mp4   ← video

    Returns dict of {feature_group:feature_name: np.array} or None.
    """
    result = {}
    extract_path = Path(extract_dir)

    # PRIMARY: Look for consolidated .npz file (WebDataset format)
    npz_path = extract_path / f'{file_id}.npz'

    if not npz_path.exists():
        # Search recursively in case of nested extraction
        npz_matches = list(extract_path.rglob(f'{file_id}.npz'))
        if npz_matches:
            npz_path = npz_matches[0]
        else:
            # FALLBACK: Try individual .npy files (S3 directory structure)
            for fg, features in features_dict.items():
                for feat in features:
                    npy_path = extract_path / fg / feat / f'{file_id}.npy'
                    if npy_path.exists():
                        try:
                            data = np.load(str(npy_path))
                            result[f'{fg}:{feat}'] = data.astype(np.float32) if data.dtype != np.float32 else data
                        except Exception as e:
                            logger.debug(f'Failed to load {npy_path}: {e}')
                    else:
                        npy_matches = list(extract_path.rglob(f'{fg}/{feat}/{file_id}.npy'))
                        if npy_matches:
                            try:
                                data = np.load(str(npy_matches[0]))
                                result[f'{fg}:{feat}'] = data.astype(np.float32) if data.dtype != np.float32 else data
                            except Exception as e:
                                logger.debug(f'Failed to load {npy_matches[0]}: {e}')
            return result if result else None

    # Load from consolidated .npz
    try:
        with np.load(str(npz_path), allow_pickle=True) as data:
            available_keys = list(data.keys())
            for fg, features in features_dict.items():
                for feat in features:
                    key_prefixed = f'{fg}:{feat}'
                    key_bare = feat

                    if key_prefixed in data:
                        arr = np.array(data[key_prefixed])
                        result[key_prefixed] = arr.astype(np.float32) if arr.dtype != np.float32 else arr
                    elif key_bare in data:
                        arr = np.array(data[key_bare])
                        result[key_prefixed] = arr.astype(np.float32) if arr.dtype != np.float32 else arr

            if not result:
                logger.debug(f'{file_id}: npz found but no selected features (keys: {available_keys[:5]}...)')
    except Exception as e:
        logger.debug(f'Failed to load {npz_path}: {e}')

    return result if result else None


def download_features_from_s3(
    file_id,
    features_dict=SELECTED_FEATURES,
    label='naturalistic',
    split='test',
    s3_base=None,
    retry_count=3,
    retry_delay=2,
    fs=None,
):
    """
    Download features for a single file_id.

    Strategy: Hugging Face tar archive (primary) → S3 direct (fallback).
    Returns tuple (file_id, {feature_key: np.array}) or (file_id, None).
    """
    if fs is None:
        config = DatasetConfig(
            label=label, split=split,
            preferred_vendors_only=True, local_dir=str(DRIVE_ROOT),
        )
        fs = SeamlessInteractionFS(config=config)

    # Validate file_id exists
    if fs._cached_filelist is not None:
        matches = fs._cached_filelist[fs._cached_filelist['file_id'] == file_id]
        if matches.empty:
            logger.debug(f'{file_id} not in filelist — skipping')
            return (file_id, None)

    # Check if already have a consolidated .npz from a previous run
    try:
        local_paths = fs.get_path_list_for_file_id_local(file_id)
        npz_candidates = [p for p in local_paths if p.endswith('.npz')]
        if npz_candidates:
            with np.load(npz_candidates[0], allow_pickle=True) as data:
                result = {}
                for fg, features in features_dict.items():
                    for feat in features:
                        for key in [f'{fg}:{feat}', feat]:
                            if key in data:
                                result[f'{fg}:{feat}'] = np.array(data[key])
                                break
                if result:
                    return (file_id, result)
    except Exception:
        pass

    # Check if archive already extracted (from a previous file_id in same archive)
    try:
        entry = fs._cached_filelist[fs._cached_filelist['file_id'] == file_id].iloc[0]
        extract_dir = Path(fs.config.local_dir) / entry['label'] / entry['split'] / f'{int(entry["batch_idx"]):04d}' / f'{int(entry["archive_idx"]):04d}'
        if extract_dir.exists():
            result = load_features_from_extracted_npy(file_id, str(extract_dir), features_dict)
            if result:
                return (file_id, result)
    except Exception:
        pass

    # PRIMARY: Download via Hugging Face archive
    extract_dir = download_hf_archive_if_needed(fs, file_id)
    if extract_dir:
        result = load_features_from_extracted_npy(file_id, extract_dir, features_dict)
        if result:
            return (file_id, result)
        else:
            logger.debug(f'{file_id}: archive extracted but features not found in {extract_dir}')

    # FALLBACK: Try S3 direct (may work for some files)
    try:
        logger.debug(f'{file_id}: trying S3 fallback...')
        fs.gather_file_id_data_from_s3(file_id, num_workers=1)
        local_paths = fs.get_path_list_for_file_id_local(file_id)
        npz_candidates = [p for p in local_paths if p.endswith('.npz')]
        if npz_candidates:
            with np.load(npz_candidates[0], allow_pickle=True) as data:
                result = {}
                for fg, features in features_dict.items():
                    for feat in features:
                        for key in [f'{fg}:{feat}', feat]:
                            if key in data:
                                result[f'{fg}:{feat}'] = np.array(data[key])
                                break
                if result:
                    return (file_id, result)
    except Exception as e:
        logger.debug(f'{file_id}: S3 fallback failed: {e}')

    return (file_id, None)


def save_features_to_drive(file_id, features_dict, npz_dir=None):
    """Save downloaded features as a bundled .npz file."""
    if npz_dir is None:
        npz_dir = DRIVE_ROOT / 'npz'
    try:
        npz_dir.mkdir(parents=True, exist_ok=True)
        npz_path = npz_dir / f'{file_id}.npz'
        np.savez_compressed(npz_path, **features_dict)
        return npz_path
    except Exception as e:
        logger.debug(f'Failed to save {file_id}: {e}')
        return None


def download_audio_from_s3(file_id, label='naturalistic', split='test',
                           s3_base=None, retry_count=3, retry_delay=2, fs=None):
    """Download a single WAV audio file. Returns (file_id, bytes) or (file_id, None)."""
    if fs is None:
        config = DatasetConfig(label=label, split=split,
                               preferred_vendors_only=True, local_dir=str(DRIVE_ROOT))
        fs = SeamlessInteractionFS(config=config)
    try:
        # Check extracted archive first
        entry = fs._cached_filelist[fs._cached_filelist['file_id'] == file_id].iloc[0]
        extract_dir = Path(fs.config.local_dir) / entry['label'] / entry['split'] / f'{int(entry["batch_idx"]):04d}' / f'{int(entry["archive_idx"]):04d}'
        wav_path = extract_dir / 'audio' / f'{file_id}.wav'
        if not wav_path.exists():
            wav_matches = list(extract_dir.rglob(f'{file_id}.wav')) if extract_dir.exists() else []
            wav_path = wav_matches[0] if wav_matches else None
        if wav_path and wav_path.exists():
            return (file_id, wav_path.read_bytes())
        # Try HF archive download
        result_dir = download_hf_archive_if_needed(fs, file_id)
        if result_dir:
            wav_matches = list(Path(result_dir).rglob(f'{file_id}.wav'))
            if wav_matches:
                return (file_id, wav_matches[0].read_bytes())
        return (file_id, None)
    except Exception as e:
        logger.debug(f'Failed audio {file_id}: {e}')
        return (file_id, None)


def download_video_from_s3(file_id, label='naturalistic', split='test',
                           s3_base=None, retry_count=3, retry_delay=2, fs=None):
    """Download a single MP4 video file. Returns (file_id, bytes) or (file_id, None)."""
    if fs is None:
        config = DatasetConfig(label=label, split=split,
                               preferred_vendors_only=True, local_dir=str(DRIVE_ROOT))
        fs = SeamlessInteractionFS(config=config)
    try:
        entry = fs._cached_filelist[fs._cached_filelist['file_id'] == file_id].iloc[0]
        extract_dir = Path(fs.config.local_dir) / entry['label'] / entry['split'] / f'{int(entry["batch_idx"]):04d}' / f'{int(entry["archive_idx"]):04d}'
        mp4_matches = list(extract_dir.rglob(f'{file_id}.mp4')) if extract_dir.exists() else []
        if mp4_matches:
            return (file_id, mp4_matches[0].read_bytes())
        result_dir = download_hf_archive_if_needed(fs, file_id)
        if result_dir:
            mp4_matches = list(Path(result_dir).rglob(f'{file_id}.mp4'))
            if mp4_matches:
                return (file_id, mp4_matches[0].read_bytes())
        return (file_id, None)
    except Exception as e:
        logger.debug(f'Failed video {file_id}: {e}')
        return (file_id, None)


def validate_wav_file(wav_bytes):
    if not wav_bytes or len(wav_bytes) < 44: return False, "File too small"
    if wav_bytes[:4] != b'RIFF': return False, "Missing RIFF header"
    if wav_bytes[8:12] != b'WAVE': return False, "Not a WAVE file"
    return True, None

def validate_mp4_bytes(mp4_bytes):
    if not mp4_bytes or len(mp4_bytes) < 12: return False, "File too small"
    if mp4_bytes[4:8] != b'ftyp': return False, "Missing ftyp box"
    return True, None

def save_audio_to_drive(file_id, audio_bytes, audio_dir=None):
    if audio_dir is None: audio_dir = DRIVE_ROOT / 'audio'
    try:
        audio_dir.mkdir(parents=True, exist_ok=True)
        path = audio_dir / f'{file_id}.wav'
        path.write_bytes(audio_bytes)
        return path
    except Exception as e:
        logger.debug(f'Failed to save audio {file_id}: {e}')
        return None

def save_video_to_drive(file_id, video_bytes, video_dir=None):
    if video_dir is None: video_dir = DRIVE_ROOT / 'video'
    try:
        video_dir.mkdir(parents=True, exist_ok=True)
        path = video_dir / f'{file_id}.mp4'
        path.write_bytes(video_bytes)
        return path
    except Exception as e:
        logger.debug(f'Failed to save video {file_id}: {e}')
        return None

print('✓ Download functions loaded (Hugging Face-first strategy)')
print(f'  Primary: Hugging Face tar archives → extract → load .npy')
print(f'  Fallback: S3 direct (gather_file_id_data_from_s3)')
print(f'  Data root: {DRIVE_ROOT}')
print(f'  Selected features: {sum(len(v) for v in SELECTED_FEATURES.values())} total')

✓ Download functions loaded (Hugging Face-first strategy)
  Primary: Hugging Face tar archives → extract → load .npy
  Fallback: S3 direct (gather_file_id_data_from_s3)
  Data root: /content/drive/MyDrive/seamless_interaction
  Selected features: 21 total


In [9]:
# ============================================================
# 8. Test Download (Single File)
# ============================================================

# Validate download pipeline with one file
if len(sampled_dyads) > 0:
    test_key = list(sampled_dyads.keys())[0]
    test_file = sampled_dyads[test_key][0]
    test_condition = sampled_condition_map.get(test_key, TARGET_LABELS[0])

    # Get metadata for this file
    meta = all_dyad_metadata.get(test_file, {
        'label': test_condition,
        'split': TARGET_SPLIT,
    })

    print(f'\U0001f9ea Test Download: {test_file}')
    print(f'   Condition: {test_condition} | Split: {meta.get("split", TARGET_SPLIT)}')

    # Initialize a temporary FS instance for the test
    test_config = DatasetConfig(
        label=meta.get('label', test_condition),
        split=meta.get('split', TARGET_SPLIT),
        preferred_vendors_only=True,
        local_dir=str(DRIVE_ROOT),
    )
    test_fs = SeamlessInteractionFS(config=test_config)

    # Check filelist first
    if test_fs._cached_filelist is not None:
        matches = test_fs._cached_filelist[test_fs._cached_filelist['file_id'] == test_file]
        if matches.empty:
            print(f'  \u2717 {test_file} NOT FOUND in filelist.csv ({len(test_fs._cached_filelist)} entries)')
            print(f'  \u2192 The file_id may not exist in the dataset')
            print(f'  \u2192 Check if label={meta.get("label")} and split={meta.get("split")} are correct')
            sample_ids = test_fs._cached_filelist['file_id'].head(3).tolist()
            print(f'  \u2192 Sample valid file_ids: {sample_ids}')
        else:
            entry = matches.iloc[0]
            print(f'   Filelist match: batch={entry["batch_idx"]}, archive={entry["archive_idx"]}')
            print(f'   S3 URLs: {len(test_fs.get_path_list_for_file_id_s3(test_file))} files')

    # Run the download
    test_file_id, test_result = download_features_from_s3(
        test_file,
        label=meta.get('label', test_condition),
        split=meta.get('split', TARGET_SPLIT),
        fs=test_fs,
    )

    if test_result is not None:
        print(f'  \u2713 Downloaded {len(test_result)} features:')
        for key in sorted(test_result.keys()):
            data = test_result[key]
            if hasattr(data, 'shape'):
                print(f'    {key}: shape={data.shape}, dtype={data.dtype}, '
                      f'range=[{np.nanmin(data):.3f}, {np.nanmax(data):.3f}]')
            else:
                print(f'    {key}: type={type(data).__name__}')

        # Validate against known sample rate
        if 'movement:emotion_valence' in test_result:
            n_frames = test_result['movement:emotion_valence'].shape[0]
            print(f'\n  Sample rate validation:')
            print(f'    Frames: {n_frames}')
            print(f'    Expected rate: {SAMPLE_RATE_HZ} Hz')
            for assumed_min in [2, 3, 5, 7, 10]:
                rate = n_frames / (assumed_min * 60)
                if 28 <= rate <= 32:
                    print(f'    \u2713 Consistent with ~{assumed_min} min interaction ({rate:.1f} Hz)')
                    break
            else:
                print(f'    \u26a0 Frame count {n_frames} doesn\'t match expected durations at {SAMPLE_RATE_HZ} Hz')
    else:
        print(f'  \u2717 Test download failed for {test_file}')
        print(f'  \u2192 Enable verbose logging: logger.setLevel(logging.DEBUG)')
        print(f'  \u2192 Or try Hugging Face batch: fs.download_batch_from_hf(batch_idx=0)')
else:
    print('\u26a0 No dyads available for test')


🧪 Test Download: V00_S0553_I00000307_P0466
   Condition: naturalistic | Split: dev
   Filelist match: batch=0, archive=35
   S3 URLs: 50 files
  ✓ Downloaded 21 features:
    movement:EmotionArousalToken: shape=(2400, 1), dtype=float32, range=[0.000, 11.000]
    movement:EmotionValenceToken: shape=(2400, 1), dtype=float32, range=[0.000, 11.000]
    movement:FAUToken: shape=(2400, 1), dtype=float32, range=[0.000, 970.000]
    movement:FAUValue: shape=(2400, 24), dtype=float32, range=[0.000, 1.994]
    movement:alignment_head_rotation: shape=(2400, 3), dtype=float32, range=[-0.431, 0.422]
    movement:alignment_translation: shape=(2400, 2, 3), dtype=float32, range=[-0.556, 0.427]
    movement:emotion_arousal: shape=(2400, 1), dtype=float32, range=[-0.179, 0.589]
    movement:emotion_scores: shape=(2400, 8), dtype=float32, range=[-3.296, 7.617]
    movement:emotion_valence: shape=(2400, 1), dtype=float32, range=[-0.342, 0.605]
    movement:expression: shape=(2400, 128), dtype=float32, ran

## Step 3: Download Movement Features (~30-60 minutes for 250 dyads)

This is the longest step. We're pulling ~5-12 GB of compressed behavioral signals from [Hugging Face archives](https://huggingface.co/datasets/facebook/seamless-interaction/tree/main). The pipeline downloads from **both conditions** (naturalistic + improvised) using separate FS instances per condition. Progress is saved automatically — you can pause and resume without losing work.

**Authentication:** If downloads fail with 403/401, run `huggingface-cli login` in a Colab cell and accept the [dataset terms](https://huggingface.co/datasets/facebook/seamless-interaction).

In [10]:
# ============================================================
# 9. Main Download Loop — Archive-Batched, Multi-Condition
# ============================================================
# Downloads features (.npz), audio (.wav), and video (.mp4) in a single pass.
# Each HF tar archive contains all modalities — no separate download needed.
# Creates one FS instance per condition label for correct archive routing.
# Budget checks are per-modality: NPZ vs TARGET_GB, audio vs AUDIO_BUDGET_GB, video vs VIDEO_BUDGET_GB.

# Initialize one SeamlessInteractionFS per condition
fs_instances = {}
for condition_label in TARGET_LABELS:
    cfg = DatasetConfig(
        label=condition_label,
        split=TARGET_SPLIT,
        preferred_vendors_only=True,
        local_dir=str(DRIVE_ROOT),
    )
    fs_instances[condition_label] = SeamlessInteractionFS(config=cfg)
    print(f'  Initialized FS for {condition_label}/{TARGET_SPLIT}')

# Check existing downloads
npz_dir = DRIVE_ROOT / 'npz'
audio_dir = DRIVE_ROOT / 'audio'
video_dir = DRIVE_ROOT / 'video'
for d in [npz_dir, audio_dir, video_dir]:
    d.mkdir(parents=True, exist_ok=True)

existing_files = {f.stem for f in npz_dir.glob('*.npz')} if npz_dir.exists() else set()
existing_audio = {f.stem for f in audio_dir.glob('*.wav')} if audio_dir.exists() else set()
existing_video = {f.stem for f in video_dir.glob('*.mp4')} if video_dir.exists() else set()
print(f'Existing: {len(existing_files)} NPZ, {len(existing_audio)} WAV, {len(existing_video)} MP4')

# Group file_ids by archive for efficient batch downloading
archive_groups = defaultdict(list)

for ikey, fids in sampled_dyads.items():
    condition = sampled_condition_map.get(ikey, TARGET_LABELS[0])
    fs_inst = fs_instances.get(condition, list(fs_instances.values())[0])

    for fid in fids:
        if fid in existing_files:
            continue
        try:
            entry = fs_inst._cached_filelist[fs_inst._cached_filelist['file_id'] == fid].iloc[0]
            archive_key = (entry['label'], entry['split'], int(entry['batch_idx']), int(entry['archive_idx']))
            archive_groups[archive_key].append((ikey, fid, condition))
        except Exception:
            logger.debug(f'Skipping {fid}: not in filelist for {condition}')

total_files = sum(len(v) for v in archive_groups.values())
print(f'Files to download: {total_files} across {len(archive_groups)} archives (skipping {len(existing_files)} existing)')

if total_files > 0:
    feature_count = sum(len(v) for v in SELECTED_FEATURES.values())
    print(f'   {feature_count} features per file x {total_files} files')
    if DOWNLOAD_AUDIO: print(f'   + audio (.wav) from same archives')
    if DOWNLOAD_VIDEO: print(f'   + video (.mp4) from same archives')
    print(f'\n  Starting download (archive-batched, multi-condition)...\n')

    downloaded = 0
    failed = 0
    audio_saved = 0
    video_saved = 0
    npz_bytes = 0           # NPZ budget tracked separately
    audio_bytes_total = 0
    video_bytes_total = 0
    _download_audio = DOWNLOAD_AUDIO  # Local copy so we can disable mid-loop
    _download_video = DOWNLOAD_VIDEO
    start_time = time.time()

    for archive_key, file_list in archive_groups.items():
        label, split, batch_idx, archive_idx = archive_key
        fs_instance = fs_instances.get(label, list(fs_instances.values())[0])

        print(f'\n  Archive {label}/{split}/{batch_idx:04d}/{archive_idx:04d} ({len(file_list)} files)')

        # Download this archive once
        extract_dir = Path(DRIVE_ROOT) / label / split / f'{batch_idx:04d}' / f'{archive_idx:04d}'
        if not (extract_dir.exists() and (any(extract_dir.rglob('*.npz')) or any(extract_dir.rglob('*.npy')))):
            extract_result = download_hf_archive_if_needed(fs_instance, file_list[0][1])
            if extract_result is None:
                print(f'    Archive download failed — skipping {len(file_list)} files')
                failed += len(file_list)
                continue
            extract_dir = Path(extract_result)

        for ikey, file_id, cond in file_list:
            try:
                # === FEATURES (.npz) ===
                features_data = load_features_from_extracted_npy(file_id, str(extract_dir))
                if features_data is None:
                    failed += 1
                    continue

                npz_path = save_features_to_drive(file_id, features_data, npz_dir)
                if npz_path and npz_path.exists():
                    npz_bytes += npz_path.stat().st_size
                    downloaded += 1
                else:
                    failed += 1
                    continue

                # === AUDIO (.wav) ===
                if _download_audio and file_id not in existing_audio:
                    wav_path = extract_dir / f'{file_id}.wav'
                    if not wav_path.exists():
                        wav_matches = list(extract_dir.rglob(f'{file_id}.wav'))
                        wav_path = wav_matches[0] if wav_matches else None
                    if wav_path and wav_path.exists():
                        import shutil
                        saved_path = audio_dir / f'{file_id}.wav'
                        shutil.copy2(str(wav_path), str(saved_path))
                        if saved_path.exists():
                            audio_bytes_total += saved_path.stat().st_size
                            audio_saved += 1

                # === VIDEO (.mp4) ===
                if _download_video and file_id not in existing_video:
                    mp4_path = extract_dir / f'{file_id}.mp4'
                    if not mp4_path.exists():
                        mp4_matches = list(extract_dir.rglob(f'{file_id}.mp4'))
                        mp4_path = mp4_matches[0] if mp4_matches else None
                    if mp4_path and mp4_path.exists():
                        video_bytes = mp4_path.read_bytes()
                        is_valid, _ = validate_mp4_bytes(video_bytes)
                        if is_valid:
                            saved = save_video_to_drive(file_id, video_bytes, video_dir)
                            if saved:
                                video_bytes_total += len(video_bytes)
                                video_saved += 1

            except Exception as e:
                failed += 1
                logger.debug(f'Error on {file_id}: {e}')

        total_all = npz_bytes + audio_bytes_total + video_bytes_total
        print(f'    {downloaded} npz ({npz_bytes/1e9:.1f}G) / {audio_saved} wav ({audio_bytes_total/1e9:.1f}G) / {video_saved} mp4 ({video_bytes_total/1e9:.1f}G) — {failed} failed')

        # Per-modality budget checks
        if npz_bytes / 1e9 > TARGET_GB:
            print(f'\n  NPZ budget reached ({npz_bytes/1e9:.1f} GB > {TARGET_GB} GB). Stopping.')
            break
        if _download_audio and audio_bytes_total / 1e9 > AUDIO_BUDGET_GB:
            print(f'  Audio budget reached ({audio_bytes_total/1e9:.1f} GB). Disabling audio for remaining files.')
            _download_audio = False
        if _download_video and video_bytes_total / 1e9 > VIDEO_BUDGET_GB:
            print(f'  Video budget reached ({video_bytes_total/1e9:.1f} GB). Disabling video for remaining files.')
            _download_video = False

    elapsed = time.time() - start_time
    total_all = npz_bytes + audio_bytes_total + video_bytes_total
    print(f'\nDownload Summary:')
    print(f'  Features (NPZ): {downloaded} files ({npz_bytes/1e9:.2f} GB)')
    print(f'  Audio (WAV):    {audio_saved} files ({audio_bytes_total/1e9:.2f} GB)')
    print(f'  Video (MP4):    {video_saved} files ({video_bytes_total/1e9:.2f} GB)')
    print(f'  Failed:         {failed}')
    print(f'  Total size:     {total_all/1e9:.2f} GB')
    print(f'  Time:           {elapsed/60:.1f} min ({elapsed/3600:.2f} hours)')
    if elapsed > 0:
        print(f'  Speed:          {total_all/1e6/elapsed:.1f} MB/s')

    for name, d in [('npz', npz_dir), ('audio', audio_dir), ('video', video_dir)]:
        if d.exists():
            count = len(list(d.iterdir()))
            size = sum(f.stat().st_size for f in d.iterdir() if f.is_file())
            print(f'    {name}/: {count} files, {size/1e9:.2f} GB')
else:
    print('  All feature files already downloaded!')


  Initialized FS for naturalistic/dev
  Initialized FS for improvised/dev
Existing: 595 NPZ, 967 WAV, 900 MP4
Files to download: 512 across 102 archives (skipping 595 existing)
   21 features per file x 512 files
   + audio (.wav) from same archives
   + video (.mp4) from same archives

  Starting download (archive-batched, multi-condition)...


  Archive naturalistic/dev/0000/0039 (11 files)
    ⬇ Downloading HF archive: naturalistic/dev/0000/0039.tar ...


naturalistic/dev/0000/0039.tar:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

    11 npz (0.1G) / 11 wav (0.4G) / 11 mp4 (0.4G) — 0 failed

  Archive naturalistic/dev/0000/0027 (11 files)
    ⬇ Downloading HF archive: naturalistic/dev/0000/0027.tar ...


naturalistic/dev/0000/0027.tar:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

    22 npz (0.3G) / 22 wav (0.8G) / 22 mp4 (0.9G) — 0 failed

  Archive naturalistic/dev/0000/0043 (13 files)
    ⬇ Downloading HF archive: naturalistic/dev/0000/0043.tar ...


...

    501 npz (7.7G) / 442 wav (17.4G) / 501 mp4 (18.6G) — 0 failed

  Archive improvised/dev/0000/0010 (2 files)
    503 npz (7.8G) / 442 wav (17.4G) / 503 mp4 (18.7G) — 0 failed

  Archive improvised/dev/0000/0003 (1 files)
    ⬇ Downloading HF archive: improvised/dev/0000/0003.tar ...


improvised/dev/0000/0003.tar:   0%|          | 0.00/1.04G [00:00<?, ?B/s]

    504 npz (7.8G) / 443 wav (17.5G) / 504 mp4 (18.7G) — 0 failed

  Archive improvised/dev/0000/0045 (3 files)
    507 npz (7.8G) / 443 wav (17.5G) / 507 mp4 (18.8G) — 0 failed

  Archive improvised/dev/0000/0041 (2 files)
    509 npz (7.8G) / 443 wav (17.5G) / 509 mp4 (18.9G) — 0 failed

  Archive improvised/dev/0000/0006 (1 files)
    510 npz (7.9G) / 443 wav (17.5G) / 510 mp4 (18.9G) — 0 failed

  Archive naturalistic/dev/0001/0015 (2 files)
    ⬇ Downloading HF archive: naturalistic/dev/0001/0015.tar ...


naturalistic/dev/0001/0015.tar:   0%|          | 0.00/1.07G [00:00<?, ?B/s]

    512 npz (7.9G) / 445 wav (17.6G) / 512 mp4 (19.0G) — 0 failed

Download Summary:
  Features (NPZ): 512 files (7.90 GB)
  Audio (WAV):    445 files (17.56 GB)
  Video (MP4):    512 files (18.98 GB)
  Failed:         0
  Total size:     44.44 GB
  Time:           49.9 min (0.83 hours)
  Speed:          14.8 MB/s
    npz/: 1107 files, 14.31 GB
    audio/: 1412 files, 55.06 GB
    video/: 1412 files, 102.50 GB


## Step 4: Validate Data Quality

Trust but verify. We check for corrupt files, missing features, NaN gaps, zero-length arrays, and dyad completeness. Only clean data enters the analysis pipeline.

In [11]:
# ============================================================
# 10. Validation & Data Quality Report
# ============================================================

npz_dir = DRIVE_ROOT / 'npz'
all_npz = sorted(list(npz_dir.glob('*.npz'))) if npz_dir.exists() else []

print(f'Data Quality Report')
print(f'{"="*60}')
print(f'  NPZ files on Drive: {len(all_npz)}')
print(f'  Total size: {sum(f.stat().st_size for f in all_npz) / 1e9:.2f} GB')

if len(all_npz) > 0:
    # Group into dyads (from disk — independent of Cell 7 registry)
    dyads_on_disk = defaultdict(list)
    for f in all_npz:
        ikey = '_'.join(f.stem.split('_')[:3])
        dyads_on_disk[ikey].append(f)

    complete_dyads_disk = {k: v for k, v in dyads_on_disk.items() if len(v) == 2}
    incomplete_disk = {k: v for k, v in dyads_on_disk.items() if len(v) != 2}

    print(f'  Complete dyads (2 files): {len(complete_dyads_disk)}')
    print(f'  Incomplete (missing participant): {len(incomplete_disk)}')

    # --- Per-condition distribution of downloaded data ---
    if 'sampled_condition_map' in dir() and len(sampled_condition_map) > 0:
        print(f'\n  Per-Condition Distribution (downloaded):')
        for condition_label in TARGET_LABELS:
            cond_keys = [k for k in complete_dyads_disk if sampled_condition_map.get(k) == condition_label]
            if not cond_keys:
                continue
            classified = condition_classified.get(condition_label, {})
            s_count = sum(1 for k in cond_keys if k in classified.get('stranger', []))
            f_count = sum(1 for k in cond_keys if k in classified.get('familiar', []))
            print(f'    {condition_label}: {len(cond_keys)} dyads (stranger={s_count}, familiar={f_count})')

            if CONDITION_CONFIG.get(condition_label, {}).get('role') == 'primary':
                if f_count == 0 or s_count == 0:
                    print(f'    *** IMBALANCE FLAG: primary condition has 0 in one class! ***')

    # Inspect feature shapes
    print(f'\n  Feature Shapes (sample of first 3 files):')
    for npz_path in all_npz[:3]:
        data = np.load(npz_path, allow_pickle=True)
        print(f'\n  {npz_path.name}:')
        for key in sorted(data.files):
            arr = data[key]
            print(
                f'    {key}: shape={arr.shape}, dtype={arr.dtype}, '
                f'range=[{np.nanmin(arr):.3f}, {np.nanmax(arr):.3f}]'
            )
        data.close()

    # Validate sample rate against known constant
    print(f'\n  Sample Rate Validation:')
    print(f'  Expected: {SAMPLE_RATE_HZ} Hz (from Seamless Interaction documentation)')

    sample_file = np.load(all_npz[0], allow_pickle=True)
    if 'movement:emotion_valence' in sample_file.files:
        n_frames = sample_file['movement:emotion_valence'].shape[0]

        matches = []
        for assumed_min in [2, 3, 5, 7, 10]:
            estimated_hz = n_frames / (assumed_min * 60)
            if 28 <= estimated_hz <= 32:
                matches.append((assumed_min, estimated_hz))

        if matches:
            for assumed_min, estimated_hz in matches:
                print(f'    {n_frames} frames / {assumed_min} min = {estimated_hz:.1f} Hz (within tolerance)')
        else:
            print(f'    {n_frames} frames: no match found within +/-2 Hz tolerance')
    else:
        print(f'    Primary feature not found in first file')

    sample_file.close()

    # Corruption check
    print(f'\nCorruption Check:')
    corrupt = 0
    suspicious = []
    for npz_path in all_npz:
        try:
            data = np.load(npz_path, allow_pickle=True)
            if 'movement:emotion_valence' in data.files:
                arr = data['movement:emotion_valence']
                if arr.shape[0] == 0:
                    corrupt += 1
                    suspicious.append((npz_path.name, 'zero-length array'))
                elif np.all(np.isnan(arr)):
                    corrupt += 1
                    suspicious.append((npz_path.name, 'all NaN'))
            if 'movement:is_valid' in data.files:
                valid_frac = data['movement:is_valid'].mean()
                if valid_frac < 0.5:
                    suspicious.append((npz_path.name, f'is_valid={valid_frac:.2f} (<50%)'))
            data.close()
        except Exception as e:
            corrupt += 1
            suspicious.append((npz_path.name, str(e)))
    print(f'  Valid: {len(all_npz) - corrupt} | Corrupt: {corrupt}')
    if suspicious:
        for name, reason in suspicious[:5]:
            print(f'    - {name}: {reason}')
        if len(suspicious) > 5:
            print(f'    ... and {len(suspicious) - 5} more')

else:
    print('  No NPZ files downloaded yet')


Data Quality Report
  NPZ files on Drive: 1107
  Total size: 14.31 GB
  Complete dyads (2 files): 541
  Incomplete (missing participant): 25

  Per-Condition Distribution (downloaded):
    naturalistic: 311 dyads (stranger=52, familiar=259)
    improvised: 125 dyads (stranger=125, familiar=0)

  Feature Shapes (sample of first 3 files):

  V00_S0062_I00000128_P0092.npz:
    movement:FAUValue: shape=(4920, 24), dtype=float32, range=[0.000, 2.000]
    movement:emotion_arousal: shape=(4920, 1), dtype=float32, range=[-0.146, 1.085]
    movement:emotion_scores: shape=(4920, 8), dtype=float32, range=[-4.885, 7.505]
    movement:emotion_valence: shape=(4920, 1), dtype=float32, range=[-0.487, 0.488]
    movement:expression: shape=(4920, 128), dtype=float32, range=[-0.804, 0.966]
    movement:gaze_encodings: shape=(4920, 2), dtype=float32, range=[-1.461, 1.424]
    movement:is_valid: shape=(4920, 1), dtype=float32, range=[1.000, 1.000]

  V00_S0062_I00000128_P0093.npz:
    movement:FAUValue: sh

## Step 5: Relationship & Condition Distribution

Verify the downloaded subset has the expected stranger/familiar balance per condition. This is the distribution the downstream coupling analysis will use.

In [12]:
# ============================================================
# 11. Relationship & Condition Distribution (Downloaded Subset)
# ============================================================

from collections import Counter

print(f'\nDownloaded Subset Distribution:')
print(f'{"="*60}')

# Show per-condition breakdown of what was actually downloaded
if 'sampled_condition_map' in dir() and len(sampled_condition_map) > 0:
    npz_dir = DRIVE_ROOT / 'npz'
    on_disk = {f.stem for f in npz_dir.glob('*.npz')} if npz_dir.exists() else set()

    # Per-condition
    for condition_label in TARGET_LABELS:
        cond_ikeys = [k for k, c in sampled_condition_map.items() if c == condition_label]
        # Filter to those actually on disk
        cond_on_disk = []
        for k in cond_ikeys:
            fids = sampled_dyads.get(k, [])
            if all(fid in on_disk for fid in fids):
                cond_on_disk.append(k)

        rels = []
        for k in cond_on_disk:
            fids = sampled_dyads.get(k, [])
            if fids:
                rels.append(get_relationship_for_dyad(fids))

        if rels:
            dist = Counter(rels)
            role = CONDITION_CONFIG.get(condition_label, {}).get('role', '?')
            print(f'  {condition_label} ({role}): {dict(dist)} = {len(cond_on_disk)} dyads')

    # Combined totals
    all_rels = []
    for k in sampled_condition_map:
        fids = sampled_dyads.get(k, [])
        if fids and all(fid in on_disk for fid in fids):
            all_rels.append(get_relationship_for_dyad(fids))
    if all_rels:
        print(f'  TOTAL: {dict(Counter(all_rels))} = {len(all_rels)} dyads')
else:
    print('  (sampled_condition_map not available — showing full CSV distribution)')
    if 'relationship' in relationships_df.columns:
        print(f'  Full CSV: {relationships_df["relationship"].value_counts().to_dict()}')

# Reference: full CSV distribution (for context)
print(f'\nFull Dataset Reference (all vendors, all splits):')
if 'relationship' in relationships_df.columns:
    rel_dist = relationships_df['relationship'].value_counts()
    print(f'  {rel_dist.to_dict()}')

if 'gender' in participants_df.columns:
    gender_dist = participants_df['gender'].value_counts()
    print(f'\nParticipant Gender Distribution:')
    print(f'  {gender_dist.to_dict()}')



Downloaded Subset Distribution:
  naturalistic (primary): {'stranger': 52, 'familiar': 259} = 311 dyads
  improvised (baseline): {'stranger': 125} = 125 dyads
  TOTAL: {'stranger': 177, 'familiar': 259} = 436 dyads

Full Dataset Reference (all vendors, all splits):
  {'stranger': 3347, 'familiar': 1751}


## Step 5b: Preview a Dyad Conversation

If video files were downloaded, we can preview a conversation between two participants directly in Colab. This is what the behavioral signals look like as raw video — the emotion, gaze, and body features in the NPZ files are extracted from footage like this.

In [ ]:
# ============================================================
# Preview: Side-by-Side Dyad Video (runs independently)
# ============================================================
# Embeds MP4 bytes as base64 data URLs — works reliably in Colab
# regardless of Drive mount / /files/ endpoint issues.

import base64
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display

# Auto-detect data root
try:
    _vdir = Path('/content/drive/MyDrive/seamless_interaction/video')
    if not _vdir.exists():
        _vdir = DRIVE_ROOT / 'video'
except NameError:
    _vdir = Path('/content/drive/MyDrive/seamless_interaction/video')

if _vdir.exists():
    all_mp4 = sorted(_vdir.glob('*.mp4'))
    print(f'Found {len(all_mp4)} video files in {_vdir}')

    if len(all_mp4) >= 2:
        # Group by interaction key (same dyad = same V_S_I prefix)
        dyad_videos = defaultdict(list)
        for mp4 in all_mp4:
            ikey = mp4.stem.rsplit('_', 1)[0]  # Drop _P{participant}
            dyad_videos[ikey].append(mp4)

        # Find first complete dyad (2 videos) with reasonable file size (<50 MB each for base64)
        sample_dyad = None
        for ikey, vids in dyad_videos.items():
            if len(vids) == 2:
                vids_sorted = sorted(vids)
                sizes_mb = [v.stat().st_size / 1e6 for v in vids_sorted]
                if all(s < 50 for s in sizes_mb):
                    sample_dyad = (ikey, vids_sorted, sizes_mb)
                    break

        if sample_dyad:
            ikey, (vid_a, vid_b), sizes_mb = sample_dyad
            print(f'\nPreviewing dyad: {ikey}')
            print(f'  Participant A: {vid_a.name} ({sizes_mb[0]:.1f} MB)')
            print(f'  Participant B: {vid_b.name} ({sizes_mb[1]:.1f} MB)')
            print(f'  Encoding as base64... (may take a moment)')

            # Read and base64-encode both videos
            b64_a = base64.b64encode(vid_a.read_bytes()).decode('ascii')
            b64_b = base64.b64encode(vid_b.read_bytes()).decode('ascii')

            html = f'''
            <div style="display: flex; gap: 10px; justify-content: center; margin: 20px 0;">
                <div style="text-align: center;">
                    <p style="font-weight: bold; font-size: 14px;">Participant A</p>
                    <video width="400" height="300" controls>
                        <source src="data:video/mp4;base64,{b64_a}" type="video/mp4">
                    </video>
                    <p style="font-size: 11px; color: #666;">{vid_a.name}</p>
                </div>
                <div style="text-align: center;">
                    <p style="font-weight: bold; font-size: 14px;">Participant B</p>
                    <video width="400" height="300" controls>
                        <source src="data:video/mp4;base64,{b64_b}" type="video/mp4">
                    </video>
                    <p style="font-size: 11px; color: #666;">{vid_b.name}</p>
                </div>
            </div>
            <p style="text-align: center; font-size: 12px; color: #888;">
                These are the raw conversations. The NPZ behavioral features (emotion, FAU, gaze)
                are extracted from video like this at 30 Hz.
            </p>
            '''
            display(HTML(html))
        else:
            print('No dyad found with both videos under 50 MB (base64 embed limit).')
            print('Showing first dyad with download links instead:')
            for ikey, vids in dyad_videos.items():
                if len(vids) == 2:
                    for v in sorted(vids):
                        print(f'  {v.name}: {v.stat().st_size/1e6:.1f} MB — {v}')
                    break
    else:
        print(f'Only {len(all_mp4)} video(s) found — need at least 2 for a dyad preview')
else:
    print(f'No video directory found at {_vdir}')
    print('Set DOWNLOAD_VIDEO = True in the config cell and re-run to download videos.')

## Step 6: Save the Download Manifest

The manifest.json is your reproducibility guarantee — documenting what you downloaded, when you downloaded it, the chosen configuration, and which files are ready for analysis. As this notebook is designed to be run multiple times, the manifest certifies exactly what has been processed.

In [15]:
# ============================================================
# 12. Save Download Manifest (for reproducibility)
# ============================================================

npz_dir = DRIVE_ROOT / 'npz'
all_npz = sorted(list(npz_dir.glob('*.npz'))) if npz_dir.exists() else []

# Rebuild dyads for manifest
dyads_found = defaultdict(list)
for f in all_npz:
    ikey = '_'.join(f.stem.split('_')[:3])
    dyads_found[ikey].append(f.stem)

complete_dyads_manifest = {k: v for k, v in dyads_found.items() if len(v) == 2}
incomplete = {k: v for k, v in dyads_found.items() if len(v) != 2}

# Count stranger/familiar in downloaded data (per condition)
condition_stats = {}
file_relationships = {}  # per-file relationship for downstream use

for condition_label in TARGET_LABELS:
    cond_fids = set()
    cond_dyad_keys = set()
    for fid_stem in [f.stem for f in all_npz]:
        ikey = '_'.join(fid_stem.split('_')[:3])
        if sampled_condition_map.get(ikey) == condition_label:
            cond_fids.add(fid_stem)
            if ikey in complete_dyads_manifest:
                cond_dyad_keys.add(ikey)

    s_count = 0
    f_count = 0
    for ikey in cond_dyad_keys:
        fids = complete_dyads_manifest[ikey]
        try:
            parts = fids[0].split('_')
            vid, sid = parts[0], int(parts[1][1:])
            match = relationships_df[
                (relationships_df['vendor_id'] == vid) &
                (relationships_df['session_id'] == sid)
            ]
            if len(match) > 0:
                rel = match.iloc[0]['relationship'].strip().lower()
                if rel == 'stranger':
                    s_count += 1
                elif rel in FAMILIAR_LABELS:
                    f_count += 1
                # else: unknown — skip, do not count in either bucket
                # Store per-file relationship
                for fid in fids:
                    file_relationships[fid] = rel
        except (ValueError, IndexError):
            pass  # Skip files with non-standard naming

    condition_stats[condition_label] = {
        'files': len(cond_fids),
        'dyads': len(cond_dyad_keys),
        'stranger': s_count,
        'familiar': f_count,
    }

# Build per-file condition mapping (critical for downstream coupling notebook)
file_conditions = {}
for f in all_npz:
    ikey = '_'.join(f.stem.split('_')[:3])
    file_conditions[f.stem] = sampled_condition_map.get(ikey, 'unknown')

# Total counts
total_stranger = sum(cs['stranger'] for cs in condition_stats.values())
total_familiar = sum(cs['familiar'] for cs in condition_stats.values())

# Hard validation BEFORE writing manifest — never persist bad data
if BALANCE_RELATIONSHIPS:
    for cond_label, cond_cfg in CONDITION_CONFIG.items():
        if cond_cfg.get('role') == 'primary':
            cs = condition_stats.get(cond_label, {})
            if cs.get('familiar', 0) == 0 or cs.get('stranger', 0) == 0:
                raise ValueError(
                    f"Balance failed for primary condition '{cond_label}': "
                    f"stranger={cs.get('stranger', 0)}, familiar={cs.get('familiar', 0)}. "
                    f"Manifest NOT written to prevent corrupt downstream reads."
                )

manifest = {
    'created': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()),
    'config': {
        's3_base': S3_BASE,
        'hf_repo': HF_REPO,
        'feature_tier': FEATURE_TIER_NAME,
        'features': {k: v for k, v in SELECTED_FEATURES.items()},
        'target_dyads': TARGET_DYADS,
        'target_gb': TARGET_GB,
        'seed': SEED,
        'sample_rate_hz': SAMPLE_RATE_HZ,
        'balance_relationships': BALANCE_RELATIONSHIPS,
        'target_labels': TARGET_LABELS,
        'target_split': TARGET_SPLIT,
        'condition_config': CONDITION_CONFIG,
        'exclude_unknown': EXCLUDE_UNKNOWN,
    },
    'results': {
        'total_files': len(all_npz),
        'complete_dyads': len(complete_dyads_manifest),
        'incomplete_dyads': len(incomplete),
        'stranger_dyads': total_stranger,
        'familiar_dyads': total_familiar,
        'balance_ratio': f'{total_stranger}:{total_familiar}',
        'conditions': condition_stats,
        'total_gb': sum(f.stat().st_size for f in all_npz) / 1e9 if len(all_npz) > 0 else 0,
    },
    'file_conditions': file_conditions,
    'file_relationships': file_relationships,
    'file_ids': sorted([f.stem for f in all_npz]),
    'dyad_keys': sorted(list(complete_dyads_manifest.keys())),
    'filelist_available': filelist_df is not None,
}

manifest_path = DRIVE_ROOT / 'pipeline_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Manifest saved: {manifest_path}')
print(f'\nPipeline Complete:')
print(f'  {len(complete_dyads_manifest)} complete dyads ready for analysis')
for cond_label, cs in condition_stats.items():
    role = CONDITION_CONFIG.get(cond_label, {}).get('role', '?')
    print(f'  {cond_label} ({role}): {cs["dyads"]} dyads (S={cs["stranger"]}, F={cs["familiar"]})')
print(f'  Total: stranger={total_stranger}, familiar={total_familiar}')
print(f'  Data location: {DRIVE_ROOT}')
print(f'  Total size: {sum(f.stat().st_size for f in all_npz) / 1e9:.2f} GB' if len(all_npz) > 0 else '  Total size: 0 GB')
print(f'\nFor multi-method coupling analysis, see dyadic_coupling_analysis.ipynb (optional downstream notebook)')
print(f'  It consumes NPZ files from the npz/ directory and reads file_conditions from the manifest.')


Manifest saved: /content/drive/MyDrive/seamless_interaction/pipeline_manifest.json

Pipeline Complete:
  541 complete dyads ready for analysis
  naturalistic (primary): 311 dyads (S=52, F=259)
  improvised (baseline): 125 dyads (S=125, F=0)
  Total: stranger=177, familiar=259
  Data location: /content/drive/MyDrive/seamless_interaction
  Total size: 14.31 GB

For multi-method coupling analysis, see dyadic_coupling_analysis.ipynb (optional downstream notebook)
  It consumes NPZ files from the npz/ directory and reads file_conditions from the manifest.
